# 08 — Interactive Visualization & Chart Selection

> **📓 Notebook · Module 04 · Intermediate**
> *Choose the right chart for the right question — every time.*

---

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

1. Use Streamlit's native charts (`st.line_chart`, `st.bar_chart`, `st.area_chart`, `st.scatter_chart`) for quick exploration.
2. Embed Matplotlib figures with `st.pyplot()` for static, publication-quality charts.
3. Embed Plotly figures with `st.plotly_chart()` for interactive dashboards.
4. Select the appropriate chart type based on the data question.
5. Apply visualization best practices: labels, titles, color, and avoiding chart junk.
6. Build interactive visualization patterns: user-selected chart types, side-by-side comparisons, dynamic data transforms.

## 📋 Prerequisites

- Completed [Notebook 07 — DataFrames, Tables & Pandas Integration](07_dataframes_tables_pandas.ipynb)
- Pandas basics (DataFrame, groupby, plot)
- Basic Matplotlib knowledge (fig, ax, plot types)
- Basic Plotly knowledge (optional — we'll learn from scratch)

---

## 📚 Concept: The Visualization Pipeline

Every visualization follows the same pipeline:

```
DATA → TRANSFORM → CHOOSE CHART → RENDER → INTERACT
 │         │            │            │          │
 ▼         ▼            ▼            ▼          ▼
Raw     Filter/       Select      Display    User
data    Aggregate     type        in app     explores
```

**The key insight:** Most of the work is in the **transform** step. A chart is only as good as the data shape you feed it.

## 🧠 Intuition: Charts Are Answers to Questions

Don't start with "I want a bar chart." Start with "What question am I answering?"

```
Question                              Chart Type
─────────────────────────────────────────────────
How does X change over time?          → Line
How do categories compare?            → Bar
What is the distribution of X?        → Histogram
Is there a relationship X vs Y?       → Scatter
What are the outliers?                → Box plot
How do parts relate to a whole?       → Pie / Treemap
How does X vary by group?             → Grouped bar / Box
```

The chart is the **answer format**. The question determines the chart.

---

## 🔧 Build It: Setup & Data Generation

Generate a realistic dataset for all our visualizations.

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np

st.set_page_config(page_title="Notebook 08", layout="wide")

# Generate realistic sales data
np.random.seed(42)
n = 365

dates = pd.date_range("2026-01-01", periods=n, freq="D")
products = ["Laptop", "Phone", "Tablet", "Monitor", "Keyboard"]
regions = ["North", "South", "East", "West"]

sales_df = pd.DataFrame({
    "Date": dates,
    "Product": np.random.choice(products, n),
    "Region": np.random.choice(regions, n),
    "Revenue": np.random.randint(5000, 50000, n).astype(float),
    "Units": np.random.randint(5, 200, n),
    "Marketing": np.random.randint(500, 10000, n).astype(float),
    "Rating": np.random.uniform(1.0, 5.0, n).round(1),
})

# Monthly aggregation for time series
monthly = sales_df.set_index("Date").resample("M").agg({
    "Revenue": "sum",
    "Units": "sum",
    "Rating": "mean",
}).reset_index()

st.header("Dataset Overview")
st.write(f"**{len(sales_df)}** daily records from **{sales_df['Date'].min().date()}** to **{sales_df['Date'].max().date()}**")
st.dataframe(sales_df.head(5), hide_index=True)

---

## 🔧 Build It: Streamlit Native Charts

Native charts require zero setup — just pass a DataFrame.

In [ ]:
st.header("Native Charts — Quick Exploration")

tab_line, tab_bar, tab_area, tab_scatter = st.tabs(["📈 Line", "📊 Bar", "🏔️ Area", "🔵 Scatter"])

with tab_line:
    st.subheader("Revenue Trend Over Time")
    st.line_chart(monthly.set_index("Date")[["Revenue", "Units"]])
    st.caption("Line charts show trends — ideal for time series data.")

with tab_bar:
    st.subheader("Revenue by Product")
    product_rev = sales_df.groupby("Product")["Revenue"].sum().sort_values(ascending=False)
    st.bar_chart(product_rev)
    st.caption("Bar charts show categorical comparisons — who's winning?")

with tab_area:
    st.subheader("Cumulative Revenue by Region")
    region_daily = sales_df.set_index("Date").groupby("Region")["Revenue"].resample("M").sum().unstack("Region")
    st.area_chart(region_daily)
    st.caption("Area charts show volume — like line charts but filled.")

with tab_scatter:
    st.subheader("Marketing vs Revenue")
    st.scatter_chart(sales_df[["Marketing", "Revenue", "Rating"]])
    st.caption("Scatter charts show relationships — is more marketing worth it?")

---

## 🔧 Build It: Matplotlib Integration

Matplotlib gives you **full control** over every visual element.

In [ ]:
import matplotlib.pyplot as plt

st.header("Matplotlib — Full Control")

# Histogram: Revenue Distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(sales_df["Revenue"], bins=40, edgecolor="black", alpha=0.7, color="#4ecdc4")
ax.set_title("Revenue Distribution", fontsize=14, fontweight="bold")
ax.set_xlabel("Revenue ($)")
ax.set_ylabel("Frequency")
ax.axvline(sales_df["Revenue"].mean(), color="red", linestyle="--", label=f"Mean: ${sales_df['Revenue'].mean():,.0f}")
ax.legend()
plt.tight_layout()
st.pyplot(fig)

st.caption("Matplotlib gives you full control: titles, labels, legends, colors, annotations.")

In [ ]:
# Multiple subplots: Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Revenue by Region
region_rev = sales_df.groupby("Region")["Revenue"].sum()
axes[0].bar(region_rev.index, region_rev.values, color=["#4ecdc4", "#ff6b6b", "#45b7d1", "#96ceb4"])
axes[0].set_title("Revenue by Region", fontweight="bold")
axes[0].set_ylabel("Total Revenue ($)")
axes[0].tick_params(axis="x", rotation=0)

# Right: Units by Product
product_units = sales_df.groupby("Product")["Units"].sum().sort_values(ascending=True)
axes[1].barh(product_units.index, product_units.values, color="#4ecdc4")
axes[1].set_title("Units Sold by Product", fontweight="bold")
axes[1].set_xlabel("Units Sold")

plt.tight_layout()
st.pyplot(fig)

In [ ]:
# Seaborn integration: Correlation heatmap
try:
    import seaborn as sns

    fig, ax = plt.subplots(figsize=(8, 6))
    numeric_cols = sales_df[["Revenue", "Units", "Marketing", "Rating"]]
    sns.heatmap(numeric_cols.corr(), annot=True, cmap="coolwarm", center=0, ax=ax,
                fmt=".2f", square=True, linewidths=0.5)
    ax.set_title("Correlation Matrix", fontweight="bold")
    plt.tight_layout()
    st.pyplot(fig)
    st.caption("Heatmaps reveal relationships between all numeric variables at once.")
except ImportError:
    st.info("Install seaborn: `pip install seaborn`")

---

## 🔧 Build It: Plotly Integration

Plotly creates **interactive** charts with hover tooltips, zoom, and export.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

st.header("Plotly — Interactive Charts")

# Line chart with hover
fig = px.line(
    monthly, x="Date", y="Revenue",
    title="Monthly Revenue Trend",
    markers=True,
)
fig.update_layout(xaxis_title="Month", yaxis_title="Revenue ($)")
st.plotly_chart(fig, use_container_width=True)
st.caption("Plotly charts are interactive: hover for details, zoom, pan, download as PNG.")

In [ ]:
# Grouped bar chart
region_product = sales_df.groupby(["Region", "Product"])["Revenue"].sum().reset_index()

fig = px.bar(
    region_product, x="Region", y="Revenue", color="Product",
    title="Revenue by Region and Product",
    barmode="group",
    color_discrete_sequence=["#4ecdc4", "#ff6b6b", "#45b7d1", "#96ceb4", "#ffeaa7"],
)
fig.update_layout(xaxis_title="Region", yaxis_title="Revenue ($)")
st.plotly_chart(fig, use_container_width=True)

In [ ]:
# Scatter with trendline
fig = px.scatter(
    sales_df, x="Marketing", y="Revenue",
    color="Region", size="Units",
    trendline="ols",
    title="Marketing Spend vs Revenue",
    color_discrete_sequence=["#4ecdc4", "#ff6b6b", "#45b7d1", "#96ceb4"],
)
fig.update_layout(xaxis_title="Marketing Spend ($)", yaxis_title="Revenue ($)")
st.plotly_chart(fig, use_container_width=True)
st.caption("Size = Units sold, Color = Region. The trendline shows the overall relationship.")

In [ ]:
# Box plot: Revenue distribution by Region
fig = px.box(
    sales_df, x="Region", y="Revenue", color="Region",
    title="Revenue Distribution by Region",
    color_discrete_sequence=["#4ecdc4", "#ff6b6b", "#45b7d1", "#96ceb4"],
)
fig.update_layout(xaxis_title="Region", yaxis_title="Revenue ($)", showlegend=False)
st.plotly_chart(fig, use_container_width=True)
st.caption("Box plots show distribution: median, quartiles, and outliers.")

In [ ]:
# Histogram: Revenue distribution
fig = px.histogram(
    sales_df, x="Revenue", nbins=40,
    title="Revenue Distribution",
    color_discrete_sequence=["#4ecdc4"],
    marginal="box",  # Add a box plot on top
)
fig.update_layout(xaxis_title="Revenue ($)", yaxis_title="Count")
st.plotly_chart(fig, use_container_width=True)

In [ ]:
# Sunburst: Hierarchical view
fig = px.sunburst(
    sales_df, path=["Region", "Product"], values="Revenue",
    title="Revenue Hierarchy: Region → Product",
)
st.plotly_chart(fig, use_container_width=True)
st.caption("Sunburst charts show hierarchy — click to drill down.")

---

## 🧪 Experiment: User-Selected Chart Types

Let users choose what they want to see.

In [ ]:
st.header("🧪 User-Selected Visualizations")

# Sidebar controls
with st.sidebar:
    st.header("📊 Chart Settings")
    chart_type = st.selectbox("Chart type", ["Line", "Bar", "Area", "Scatter"])
    x_col = st.selectbox("X axis", ["Date", "Product", "Region"])
    y_col = st.selectbox("Y axis", ["Revenue", "Units", "Rating"])
    color_by = st.selectbox("Color by", ["None", "Product", "Region"])
    show_trendline = st.checkbox("Show trendline (scatter only)")

# Transform data based on selection
if x_col == "Date":
    chart_data = sales_df.set_index("Date").resample("W")[[y_col]].sum().reset_index()
else:
    chart_data = sales_df.groupby(x_col)[[y_col]].sum().reset_index()

# Render selected chart
color_param = color_by if color_by != "None" else None

if chart_type == "Line":
    fig = px.line(chart_data, x=x_col, y=y_col, color=color_param, title=f"{y_col} by {x_col}")
elif chart_type == "Bar":
    fig = px.bar(chart_data, x=x_col, y=y_col, color=color_param, title=f"{y_col} by {x_col}")
elif chart_type == "Area":
    fig = px.area(chart_data, x=x_col, y=y_col, color=color_param, title=f"{y_col} by {x_col}")
else:
    fig = px.scatter(chart_data, x=x_col, y=y_col, color=color_param,
                     title=f"{y_col} by {x_col}")
    if show_trendline:
        fig = px.scatter(chart_data, x=x_col, y=y_col, color=color_param,
                         trendline="ols", title=f"{y_col} by {x_col}")

st.plotly_chart(fig, use_container_width=True)

---

## 🧪 Experiment: Side-by-Side Comparison

Compare two views of the same data.

In [ ]:
st.header("🧪 Side-by-Side Comparison")

col1, col2 = st.columns(2)

with col1:
    st.subheader("Top Products by Revenue")
    top_products = sales_df.groupby("Product")["Revenue"].sum().sort_values(ascending=True)
    fig1 = px.bar(x=top_products.values, y=top_products.index, orientation="h",
                  title="Revenue by Product")
    fig1.update_layout(xaxis_title="Revenue ($)", yaxis_title="Product")
    st.plotly_chart(fig1, use_container_width=True)

with col2:
    st.subheader("Top Products by Units")
    top_units = sales_df.groupby("Product")["Units"].sum().sort_values(ascending=True)
    fig2 = px.bar(x=top_units.values, y=top_units.index, orientation="h",
                  title="Units by Product", color_discrete_sequence=["#ff6b6b"])
    fig2.update_layout(xaxis_title="Units Sold", yaxis_title="Product")
    st.plotly_chart(fig2, use_container_width=True)

st.caption("Revenue and units tell different stories — a product might sell many units but at lower revenue.")

---

## 🧪 Experiment: Dynamic Data Transforms

Let users change how the data is aggregated.

In [ ]:
st.header("🧪 Dynamic Aggregation")

agg_func = st.selectbox("Aggregation function", ["Sum", "Mean", "Median", "Count"])
agg_map = {"Sum": "sum", "Mean": "mean", "Median": "median", "Count": "count"}

result = sales_df.groupby("Region")["Revenue"].agg(agg_map[agg_func]).reset_index()
result.columns = ["Region", "Revenue"]

fig = px.bar(result, x="Region", y="Revenue",
             title=f"Revenue by Region ({agg_func})",
             color="Region",
             color_discrete_sequence=["#4ecdc4", "#ff6b6b", "#45b7d1", "#96ceb4"])
fig.update_layout(showlegend=False)
st.plotly_chart(fig, use_container_width=True)

---

## ⚠️ Common Visualization Mistakes

### Mistake 1: Line Chart for Categorical Data

```python
# ❌ Line chart implies continuity between categories
st.line_chart(region_sales)  # North → South → East → West? Nonsense!

# ✅ Bar chart for discrete categories
st.bar_chart(region_sales)
```

### Mistake 2: Pie Chart with Too Many Slices

```python
# ❌ 15 slices — can't distinguish colors
fig = px.pie(df, names="Category", values="Revenue")

# ✅ Top 5 + "Other"
top5 = df.nlargest(5, "Revenue")
other_rev = df[~df.index.isin(top5.index)]["Revenue"].sum()
chart_df = pd.concat([top5, pd.DataFrame({"Category": ["Other"], "Revenue": [other_rev]})])
fig = px.pie(chart_df, names="Category", values="Revenue")
```

### Mistake 3: No Titles or Labels

```python
# ❌ What am I looking at?
st.line_chart(df[["Revenue"]])

# ✅ Clear context
st.subheader("Monthly Revenue Trend")
st.line_chart(monthly.set_index("Date")[["Revenue"]])
```

### Mistake 4: Misleading Y-Axis

```python
# ❌ Bar chart with truncated Y-axis exaggerates differences
fig.update_yaxes(range=[50000, 70000])

# ✅ Start from zero for bar charts
fig.update_yaxes(range=[0, None])
```

---

## 🔍 Debugging Tips

| Symptom | Likely Cause | Fix |
|---|---|---|
| Chart doesn't fill width | Missing `use_container_width` | Add `use_container_width=True` |
| Matplotlib figure cut off | Missing `tight_layout()` | Add `plt.tight_layout()` before `st.pyplot()` |
| Plotly chart blank | Wrong data shape | Check x/y columns exist in DataFrame |
| Native chart looks wrong | Data not in right format | Use `set_index()` for datetime index |
| Too many bars | Too many categories | Filter top N or group small categories |
| Colors inconsistent | No color parameter | Use `color_discrete_sequence` in Plotly |
| Chart not updating | Data not changing with filters | Ensure filter logic is above chart code |

---

## ✅ Best Practices

1. **Start with the question** — "What am I showing?" determines the chart type.
2. **Native charts for exploration**, Matplotlib for static analysis, Plotly for dashboards.
3. **Always `use_container_width=True`** with Plotly charts.
4. **Always add titles and axis labels** — never assume the user knows what they're seeing.
5. **Use consistent colors** across related charts.
6. **Avoid chart junk** — fewer elements, cleaner insight.
7. **Start bar chart Y-axes from zero** — don't mislead.
8. **Filter before you visualize** — don't show everything at once.
9. **Use tabs** to show multiple views without overwhelming the user.
10. **Side-by-side comparisons** reveal insights that single charts miss.

---

## ✏️ Exercises

### Exercise 1: Chart Selection
For each of these questions, choose the right chart type and implement it:
1. How did revenue change over the past 12 months? (Line)
2. Which product has the most units sold? (Bar)
3. What is the distribution of daily revenue? (Histogram)
4. Is there a relationship between marketing spend and revenue? (Scatter)
5. How does revenue vary by region? (Box plot)

### Exercise 2: Interactive Dashboard
Build a visualization dashboard with:
- Sidebar controls: chart type selector, X-axis, Y-axis, color-by
- A Plotly chart that updates based on selections
- A Matplotlib histogram in a second tab
- Side-by-side comparison of two aggregations

### Exercise 3: Chart Makeover
Take a poorly designed chart (no labels, wrong type, misleading axis) and fix it step by step. Document each improvement.

## 🚀 Challenge Problem

Build a **Complete Visualization Dashboard** that:
1. Generates a 365-day sales dataset with Product, Region, Revenue, Units, Marketing, Rating
2. Has sidebar controls for: chart type (native/Matplotlib/Plotly), X-axis, Y-axis, color-by, aggregation
3. Shows a Plotly interactive chart that updates dynamically
4. Includes a Matplotlib subplot panel (2 charts side by side)
5. Has a tabbed view: Trends | Distributions | Relationships
6. Shows a correlation heatmap in one tab
7. Uses consistent color palette across all charts
8. Every chart has a title and axis labels

Use `np.random.seed(42)` for reproducibility.

---

## 📌 Key Takeaways

1. **The question determines the chart** — trends→line, categories→bar, distribution→histogram, relationships→scatter.
2. **Native charts** are for **quick exploration** — zero setup.
3. **Matplotlib** is for **static, publication-quality** figures with full control.
4. **Plotly** is for **interactive dashboards** — hover, zoom, export. Always use `use_container_width=True`.
5. **Transform before you visualize** — the data shape determines the chart's effectiveness.
6. **Label everything** — titles, axis labels, legends. Never assume the user knows what they're looking at.
7. **Avoid chart junk** — fewer elements, cleaner insight.

---

## 📚 Further Reading

- [Streamlit Chart Elements](https://docs.streamlit.io/develop/api-reference/charts)
- [st.plotly_chart Reference](https://docs.streamlit.io/develop/api-reference/charts/st.plotly_chart)
- [st.pyplot Reference](https://docs.streamlit.io/develop/api-reference/charts/st.pyplot)
- [Plotly Express Documentation](https://plotly.com/python/plotly-express/)
- [Matplotlib Tutorials](https://matplotlib.org/stable/tutorials/index.html)

---

## 🔗 Related Materials

- 📖 Reading: [07 — Data Display: DataFrames, Tables & Pandas Integration](../readings/07_data_display_dataframes.md)
- 📖 Reading: [08 — Visualization with Streamlit, Matplotlib & Plotly](../readings/08_visualization_matplotlib_plotly.md)
- 📓 Notebook: [07 — DataFrames, Tables & Pandas Integration](07_dataframes_tables_pandas.ipynb)
- ✏️ Exercise: [07 — Data Display Challenges](../exercises/07_data_display_challenges.py)
- ✏️ Exercise: [08 — Visualization Workshop](../exercises/08_visualization_workshop.py)
- 🖥️ Demo App: [07 — Data Display Demo](../apps/07_data_display_demo.py)
- 📝 Quiz: [04 — DataFrames & Visualization](../quizzes/04_dataframes_visualization.md)